# Exploratory Data Analysis — EDM Analytics Pipeline

This notebook explores the **synthetic** student academic dataset produced by `src/data_generator.py` and processed by the rest of the pipeline.

> All data used here is synthetic. No real students or institutions are represented.

Run the full pipeline first from the project root so the files referenced below exist:
```
python -m src.main
```

In [ ]:
import sys
from pathlib import Path

# Allow imports from the project's src/ package when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load the cleaned, featured, and risk-scored data

In [ ]:
features_path = PROJECT_ROOT / "data" / "processed" / "featured_student_academic_data.csv"
risk_path = PROJECT_ROOT / "data" / "processed" / "student_risk_scores.csv"

df = pd.read_csv(features_path).merge(pd.read_csv(risk_path), on="student_id")
print(df.shape)
df.head()

## 2. Distribution of core variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df["attendance_rate"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Attendance Rate")
sns.histplot(df["assessment_average"], bins=30, kde=True, ax=axes[1])
axes[1].set_title("Assessment Average")
sns.histplot(df["final_exam_score"], bins=30, kde=True, ax=axes[2])
axes[2].set_title("Final Exam Score")
plt.tight_layout()
plt.show()

## 3. Relationships with final exam score

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, ["attendance_rate", "study_hours_per_week", "previous_gpa"]):
    sns.regplot(data=df, x=col, y="final_exam_score", scatter_kws={"alpha": 0.2, "s": 10}, line_kws={"color": "red"}, ax=ax)
    ax.set_title(f"{col} vs final_exam_score")
plt.tight_layout()
plt.show()

## 4. Correlation matrix (pre-exam variables + outcome)

Correlation only — not a causal claim about any real population.

In [ ]:
corr_cols = [
    "attendance_rate", "study_hours_per_week", "assessment_average",
    "performance_trend", "previous_gpa", "risk_score", "final_exam_score",
]
corr = df[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix (Pre-Exam Variables + Risk Score + Final Exam Score)")
plt.tight_layout()
plt.show()

## 5. Risk level distribution and characteristics

In [ ]:
print(df["risk_level"].value_counts())
df.groupby("risk_level")[["attendance_rate", "assessment_average", "previous_gpa", "final_exam_score"]].mean().round(2)

## 6. Next steps

The full statistical summary, model performance comparison, and all
saved chart images referenced in the README live under `data/reports/`
once `python -m src.main` (or `src/analytics.py` directly) has been run.
See `docs/methodology.md` for the full methodology behind each metric
shown above.